# 004 Structured Output

这是 LangChain 学习线的第四份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/structured-output
- https://docs.langchain.com/oss/python/langchain/agents
- https://docs.langchain.com/oss/python/langchain-models

学习目标：

1. 理解为什么“提示模型返回 JSON”不等于可靠结构化输出
2. 用 Pydantic 定义结构化 schema
3. 理解 LangChain 的 `response_format`
4. 区分 ProviderStrategy 和 ToolStrategy
5. 把 structured output 映射到本仓库 planner action

---

## 1. 为什么需要 Structured Output

前面我们在 Harness runtime 里遇到过一个问题：

```text
提示模型：请返回 JSON
模型可能返回：自然语言、Markdown 代码块、半截 JSON、字段缺失 JSON
```

这会让 `_json_plan_from_text()` 很脆弱。

Structured output 的目标是：

```text
把模型输出约束成应用能直接使用的结构化数据。
```

在 LangChain agent 中，可以通过 `response_format` 指定 schema，最终结构化结果会出现在 `structured_response` 里。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 先看 prompt-only JSON 的脆弱性

只靠 `json.loads(...)`，要求模型输出必须刚好是合法 JSON。

In [1]:
import json

samples = [
    '{"action": "delegate", "role": "research"}',
    '我建议走 research subagent。',
    '```json\n{"action": "delegate", "role": "research"}\n```',
]

for text in samples:
    try:
        parsed = json.loads(text)
        print("OK:", parsed)
    except json.JSONDecodeError as exc:
        print("FAIL:", repr(text), "=>", exc)

OK: {'action': 'delegate', 'role': 'research'}
FAIL: '我建议走 research subagent。' => Expecting value: line 1 column 1 (char 0)
FAIL: '```json\n{"action": "delegate", "role": "research"}\n```' => Expecting value: line 1 column 1 (char 0)


## 3. 用 Pydantic 定义 planner schema

我们把本仓库 planner 的 action 抽象成一个 Pydantic 模型。

这不是直接替换当前代码，而是先学习 structured output 的设计形态。

In [2]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class PlannerDecision(BaseModel):
    """A normalized planner decision for a Harness-style agent."""

    action: Literal["answer", "tool", "delegate", "delegate_batch"] = Field(
        description="The next action selected by the planner."
    )
    reason: str = Field(description="Short reason for the decision.")
    role: Literal["research", "verification", "implementation"] | None = Field(
        default=None,
        description="Subagent role when action is delegate.",
    )
    tool_name: str | None = Field(default=None, description="Tool name when action is tool.")
    allowed_paths: list[str] = Field(default_factory=list, description="Allowed repo paths for subagent work.")


print(json.dumps(PlannerDecision.model_json_schema(), ensure_ascii=False, indent=2))

{
  "description": "A normalized planner decision for a Harness-style agent.",
  "properties": {
    "action": {
      "description": "The next action selected by the planner.",
      "enum": [
        "answer",
        "tool",
        "delegate",
        "delegate_batch"
      ],
      "title": "Action",
      "type": "string"
    },
    "reason": {
      "description": "Short reason for the decision.",
      "title": "Reason",
      "type": "string"
    },
    "role": {
      "anyOf": [
        {
          "enum": [
            "research",
            "verification",
            "implementation"
          ],
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Subagent role when action is delegate.",
      "title": "Role"
    },
    "tool_name": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "descrip

## 4. 本地验证：字段不对会失败

Pydantic 的价值是：结构不对时，应用能得到明确错误，而不是拿到一段看似合理的自然语言。

In [3]:
valid_data = {
    "action": "delegate",
    "role": "research",
    "reason": "需要先做只读代码调查",
    "allowed_paths": ["app/agents/harness.py"],
}

invalid_data = {
    "action": "random_action",
    "reason": "模型编造了不存在的 action",
}

print(PlannerDecision.model_validate(valid_data))

try:
    PlannerDecision.model_validate(invalid_data)
except ValidationError as exc:
    print("Validation failed:")
    print(exc)

action='delegate' reason='需要先做只读代码调查' role='research' tool_name=None allowed_paths=['app/agents/harness.py']
Validation failed:
1 validation error for PlannerDecision
action
  Input should be 'answer', 'tool', 'delegate' or 'delegate_batch' [type=literal_error, input_value='random_action', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 5. LangChain 的 response_format

LangChain agent 的 `create_agent(...)` 支持 `response_format`。

官方文档里有三种常见写法：

- 直接传 schema 类型：LangChain 自动选择策略
- `ProviderStrategy`：使用模型供应商原生 structured output
- `ToolStrategy`：通过工具调用模拟结构化输出

如果模型和供应商支持原生结构化输出，ProviderStrategy 通常更可靠；否则可以退回 ToolStrategy。

In [4]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

provider_strategy = ProviderStrategy(PlannerDecision)
tool_strategy = ToolStrategy(PlannerDecision)

print(provider_strategy)
print(tool_strategy)

ProviderStrategy(schema=<class '__main__.PlannerDecision'>, schema_spec=_SchemaSpec(schema=<class '__main__.PlannerDecision'>, name='PlannerDecision', description='A normalized planner decision for a Harness-style agent.', schema_kind='pydantic', json_schema={'description': 'A normalized planner decision for a Harness-style agent.', 'properties': {'action': {'description': 'The next action selected by the planner.', 'enum': ['answer', 'tool', 'delegate', 'delegate_batch'], 'title': 'Action', 'type': 'string'}, 'reason': {'description': 'Short reason for the decision.', 'title': 'Reason', 'type': 'string'}, 'role': {'anyOf': [{'enum': ['research', 'verification', 'implementation'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Subagent role when action is delegate.', 'title': 'Role'}, 'tool_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Tool name when action is tool.', 'title': 'Tool Name'}, 'allowed_paths': {'descripti

## 6. 可选：用 ChatOpenAI.with_structured_output

如果你已经配置好 `.env`，可以直接让模型按 `PlannerDecision` 输出。

注意：不同模型和 OpenAI 兼容网关对 structured output 支持不同。失败时不要先怀疑 schema，先确认网关是否支持。

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

if not OPENAI_API_KEY:
    structured_model = None
    print("Skip live structured output because OPENAI_API_KEY is missing.")
else:
    model = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    structured_model = model.with_structured_output(PlannerDecision)
    print("Structured model ready:", OPENAI_MODEL)

Structured model ready: qwq


In [6]:
if structured_model is None:
    print("Skip live invoke.")
else:
    try:
        decision = structured_model.invoke(
            "用户问：帮我检查 app/agents/harness.py 里 approval 恢复流程。请给出下一步 planner decision。"
        )
        print(decision)
        print(type(decision))
    except Exception as exc:
        print("Structured output call failed. Check whether the model gateway supports structured output.")
        print(type(exc).__name__, exc)

action='answer' reason='用户未提供文件内容或上下文，无法直接分析。需引导用户提供关键代码片段与运行状态，并说明下一步决策的生成逻辑，保持专业、可执行。' role='research' tool_name='none' allowed_paths=[]
<class '__main__.PlannerDecision'>


## 7. 可选：在 create_agent 中使用 response_format

LangChain agent 的 structured output 会放在最终 state 的 `structured_response` 字段里。

这和本仓库当前 `plan` dict 很像，但区别是：LangChain 会帮你做 schema 层验证。

In [7]:
from langchain.agents import create_agent

if not OPENAI_API_KEY:
    agent = None
    print("Skip create_agent because OPENAI_API_KEY is missing.")
else:
    model = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    agent = create_agent(
        model=model,
        tools=[],
        response_format=PlannerDecision,
        system_prompt="你是一个 Harness 风格 planner，只做下一步决策。",
    )
    print("Agent with structured response ready")

Agent with structured response ready


In [8]:
if agent is None:
    print("Skip agent invoke.")
else:
    try:
        result = agent.invoke({
            "messages": [
                {"role": "user", "content": "帮我检查 app/agents/harness.py 里 approval 恢复流程涉及哪些方法"}
            ]
        })
        print(result.get("structured_response"))
    except Exception as exc:
        print("Agent structured output failed. Check model support or use ToolStrategy explicitly.")
        print(type(exc).__name__, exc)

action='tool' reason='To identify methods involved in the approval recovery flow, I first need to read the content of the file `app/agents/harness.py`.' role=None tool_name='read_file' allowed_paths=[]


## 8. 回到本仓库：planner 可以如何升级

当前 Harness planner 主要靠：

```text
OpenAI tool call -> action=tool
合法 JSON 文本 -> action=delegate/delegate_batch/answer/tool
其他文本 -> action=answer
```

这适合教学，但工程上可以继续升级为：

```text
PlannerDecision schema
  -> model structured output
  -> Pydantic validation
  -> policy guard
  -> execution
```

注意：structured output 只解决“输出结构稳定”，不解决权限问题。权限仍然要由 Harness policy 判断。

## 9. 本讲小结

这一讲记住四点：

1. prompt-only JSON 不可靠，因为模型可能输出自然语言或 Markdown。
2. Pydantic schema 可以明确字段、枚举和校验规则。
3. LangChain `response_format` 可以让 agent 返回 `structured_response`。
4. structured output 不替代 policy；它只是让 planner 输出更稳定。

下一讲建议学习：

- LangChain agent 的控制流
- 什么时候调用工具，什么时候回答
- 如何和本仓库 Harness Query Loop 对比